In [2]:
import threading
import time
from socket import *

# =====================================================================
# 0. PREPARATION: Create a dummy file in Colab to test the system
# =====================================================================
TEST_FILENAME = "example.txt"
with open(TEST_FILENAME, "w") as f:
    f.write("Hello from Google Colab! The TCP socket transfer worked perfectly.")

# Network Configuration
SERVER_NAME = "127.0.0.1"
SERVER_PORT = 12005  # Using a unique port to avoid conflicts

# =====================================================================
# 1. SERVER CODE (Runs in a background thread)
# =====================================================================
def run_server():
    # Create, configure, and bind TCP socket
    serverSocket = socket(AF_INET, SOCK_STREAM)
    serverSocket.setsockopt(SOL_SOCKET, SO_REUSEADDR, 1) # Allows quick restarts
    serverSocket.bind((SERVER_NAME, SERVER_PORT))
    serverSocket.listen(1)
    print("[SERVER] Ready to receive connections...\n")

    while True:
        try:
            # Accept client connection
            connectionSocket, addr = serverSocket.accept()

            # Receive the requested filename
            sentence = connectionSocket.recv(1024).decode().strip()

            # Read file if present, else catch error safely
            try:
                with open(sentence, "r") as file:
                    l = file.read(1024)
                    connectionSocket.send(l.encode())
            except FileNotFoundError:
                connectionSocket.send(f"Error: File '{sentence}' not found on server.".encode())

            # Clean up client connection
            connectionSocket.close()
        except Exception as e:
            # Prevent background crash loop
            break

# Start the server thread cleanly in the background
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Short delay to give the server thread a moment to bind and launch
time.sleep(0.5)

# =====================================================================
# 2. CLIENT CODE (Runs in the main foreground thread)
# =====================================================================
def run_client():
    print(f"[CLIENT] Available file to test: '{TEST_FILENAME}'")
    filename_to_request = input("[CLIENT] Enter file name: ")

    # Setup socket and connect
    clientSocket = socket(AF_INET, SOCK_STREAM)
    clientSocket.connect((SERVER_NAME, SERVER_PORT))

    # Send request and await transmission
    clientSocket.send(filename_to_request.encode())
    filecontents = clientSocket.recv(1024).decode()

    # Display results
    print("\n--- [CLIENT] Received From Server ---")
    print(filecontents)
    print("-------------------------------------\n")

    clientSocket.close()

# Execute the client action
run_client()


[SERVER] Ready to receive connections...

[CLIENT] Available file to test: 'example.txt'
[CLIENT] Enter file name: example.txt

--- [CLIENT] Received From Server ---
Hello from Google Colab! The TCP socket transfer worked perfectly.
-------------------------------------

